# Padding et masque : réunir des durées sans changer la moyenne

Compléter un lot, puis compter uniquement les positions réellement observées.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#padding-et-collate).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Notre chargeur empilait des lignes de même taille. Un son de deux trames et un autre d’une seule ne forment pourtant pas une matrice rectangulaire. Changeons d’expérience pour isoler ce problème : A contient [2,4], B contient [6], en unités arbitraires. Ce ne sont pas les RMS de nos trois extraits.

Chaque trame porte une seule mesure : les séquences ont les formes (2,1) et (1,1). pad_sequence ajoute une trame nulle à B. batch_first=True place les extraits en premier ; le lot devient (2,2,1), avec les axes extraits/temps/mesures.

Le remplissage n’est pas une observation. Le masque compare les positions 0,1 aux longueurs 2,1 : [[True,True],[True,False]]. Il vient des longueurs, car un véritable signal peut lui aussi contenir zéro.

La fonction assembler est notre collate_fn : elle remplace l’assemblage par défaut lorsque le chargeur reçoit sa liste d’exemples. Le programme complet en montre le dépaquetage. Regardons surtout la mesure obtenue après l’assemblage : une moyenne naïve donnerait 3 à A… et 3 à B.

unsqueeze(-1) aligne le masque sur l’axe des mesures : (2,2,1). La somme valide vaut 6 pour chaque extrait, mais les comptes valent 2 et 1. Il faut masquer le numérateur et corriger le dénominateur : on retrouve 3 et 6, quelle que soit la quantité de remplissage.

| Extrait | Somme valide | Compte | Moyenne |
| --- | --- | --- | --- |
| A | 2+4=6 | 2 | 3 |
| B | 6 | 1 | 6 |

La moyenne naïve de B divisait par deux en comptant le remplissage.

Pour une erreur quadratique moyenne, le compte doit aussi correspondre aux valeurs additionnées. Comparons ici les mesures à zéro. Si chaque trame possède D descripteurs, elle apporte D carrés d’erreur : il faut compter trames valides × D.

Ici D=1 : (2²+4²+6²)/3=56/3≈18,67. Ajouter du padding ne doit changer ni cette somme ni ce compte. Nous supposons des valeurs finies : multiplier un NaN par zéro ne le rendrait pas valide.

## Démonstration 1 · Assembler, masquer et calculer les bons dénominateurs

Retrouver les objets et exécuter les fragments du chapitre dans leur contexte complet.

### Préparer les bibliothèques

Une mesure par trame, dans un exemple indépendant en unités arbitraires.

In [3]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
sequence_a = torch.tensor([[2.], [4.]])
sequence_b = torch.tensor([[6.]])
exemples = [(sequence_a, 0), (sequence_b, 1)]



In [5]:
observer('sequence_a', sequence_a)
observer('sequence_b', sequence_b)
observer('exemples', exemples)

sequence_a forme (2, 1) type torch.float32 requires_grad False
  valeurs : [[2.0], [4.0]]
sequence_b forme (1, 1) type torch.float32 requires_grad False
  valeurs : [[6.0]]
exemples : 2 éléments ; formes des tenseurs : []


### Définir la règle avant de l’utiliser

zip sépare les séquences et classes. Les longueurs déterminent le masque après padding.

In [6]:
def assembler(exemples):
    sequences, annotations = zip(*exemples)
    longueurs = torch.tensor([len(sequence) for sequence in sequences])
    valeurs = pad_sequence(sequences, batch_first=True, padding_value=0.)
    positions = torch.arange(valeurs.shape[1])
    masque = positions[None, :] < longueurs[:, None]
    cibles = torch.tensor(annotations, dtype=torch.long)
    return valeurs, cibles, masque



### Calculer l’état suivant

collate_fn reçoit les exemples du lot puis rend les trois objets attendus.

In [7]:
loader = DataLoader(exemples, batch_size=2, collate_fn=assembler, num_workers=0)
batch, cibles, masque = next(iter(loader))


In [8]:
observer('batch', batch)
observer('cibles', cibles)
observer('masque', masque)

batch forme (2, 2, 1) type torch.float32 requires_grad False
  valeurs : [[[2.0], [4.0]], [[6.0], [0.0]]]
cibles forme (2,) type torch.int64 requires_grad False
  valeurs : [0, 1]
masque forme (2, 2) type torch.bool requires_grad False
  valeurs : [[True, True], [True, False]]


### Calculer l’état suivant

Le masque est aligné avec les mesures ; les comptes ne sont pas la longueur maximale.

In [9]:
validite = masque.unsqueeze(-1)
sommes = (batch * validite).sum(dim=1)
comptes = validite.sum(dim=1)
moyennes = sommes / comptes


In [10]:
observer('validite', validite)
observer('sommes', sommes)
observer('comptes', comptes)
observer('moyennes', moyennes)

validite forme (2, 2, 1) type torch.bool requires_grad False
  valeurs : [[[True], [True]], [[True], [False]]]
sommes forme (2, 1) type torch.float32 requires_grad False
  valeurs : [[6.0], [6.0]]
comptes forme (2, 1) type torch.int64 requires_grad False
  valeurs : [[2], [1]]
moyennes forme (2, 1) type torch.float32 requires_grad False
  valeurs : [[3.0], [6.0]]


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [11]:
naives = batch.mean(dim=1)


In [12]:
observer('naives', naives)

naives forme (2, 1) type torch.float32 requires_grad False
  valeurs : [[3.0], [3.0]]


### Calculer l’état suivant

La cible choisie est zéro. Le dénominateur inclut les mesures de chaque trame valide.

In [13]:
cible_zero = torch.zeros_like(batch)
carres_valides = (batch - cible_zero).square() * validite
denominateur = masque.sum() * batch.shape[-1]
mse = carres_valides.sum() / denominateur


In [14]:
observer('cible_zero', cible_zero)
observer('carres_valides', carres_valides)
observer('denominateur', denominateur)
observer('mse', mse)

cible_zero forme (2, 2, 1) type torch.float32 requires_grad False
  valeurs : [[[0.0], [0.0]], [[0.0], [0.0]]]
carres_valides forme (2, 2, 1) type torch.float32 requires_grad False
  valeurs : [[[4.0], [16.0]], [[36.0], [0.0]]]
denominateur forme () type torch.int64 requires_grad False
  valeurs : 3
mse forme () type torch.float32 requires_grad False
  valeurs : 18.66666603088379


### Lire les sorties

Les sorties rendent visibles la case ajoutée et son exclusion.

In [15]:
print("lot :", batch.squeeze(-1).tolist(), "forme", tuple(batch.shape))
print("masque :", masque.tolist())
print("sommes :", sommes.squeeze(-1).tolist())
print("comptes :", comptes.squeeze(-1).tolist())


lot : [[2.0, 4.0], [6.0, 0.0]] forme (2, 2, 1)
masque : [[True, True], [True, False]]
sommes : [6.0, 6.0]
comptes : [2, 1]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [16]:
print("moyennes masquees :", moyennes.squeeze(-1).tolist())
print("moyennes naives :", naives.squeeze(-1).tolist())
print("MSE vers zero :", round(mse.item(), 6), "sur", denominateur.item(), "valeurs")


moyennes masquees : [3.0, 6.0]
moyennes naives : [3.0, 3.0]
MSE vers zero : 18.666666 sur 3 valeurs


In [17]:
assert bool(tuple(batch.shape) == (2,2,1) and masque.tolist() == [[True,True],[True,False]] and cibles.tolist() == [0,1]), 'padding-et-collate/main: tuple(batch.shape) == (2,2,1) and masque.tolist() == [[True,True],[True,False]] and cibles.tolist() == [0,1]'
assert bool(moyennes.squeeze(-1).tolist() == [3.,6.] and naives.squeeze(-1).tolist() == [3.,3.]), 'padding-et-collate/main: moyennes.squeeze(-1).tolist() == [3.,6.] and naives.squeeze(-1).tolist() == [3.,3.]'
assert bool(comptes.squeeze(-1).tolist() == [2,1] and int(denominateur) == 3), 'padding-et-collate/main: comptes.squeeze(-1).tolist() == [2,1] and int(denominateur) == 3'
assert bool(abs(mse.item()-56/3) < 2e-6), 'padding-et-collate/main: abs(mse.item()-56/3) < 2e-6'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

A contient [2,4] et B [6], stockés après padding comme [[2,4],[6,0]]. Écrivez les deux moyennes qui ne comptent que les trames valides.

In [18]:
exo_batch = torch.tensor([[[2.],[4.]],[[6.],[0.]]])
exo_masque = torch.tensor([[True,True],[True,False]])
exo_validite = exo_masque.unsqueeze(-1)
exo_moyennes = ...  # Remplacer les points par votre réponse.
if exo_moyennes is Ellipsis or exo_moyennes is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(exo_moyennes,torch.tensor([[3.],[6.]]))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La somme vaut six dans chaque extrait, mais les comptes sont deux et un. Gardez l’axe des mesures.

</details>

### Exercice 2

Chaque trame valide porte maintenant D=2 descripteurs. Quel dénominateur utiliser pour la MSE par valeur vers zéro ?

In [19]:
exo_batch = torch.tensor([[[2.,20.],[4.,40.]],[[6.,60.],[0.,0.]]])
exo_masque = torch.tensor([[True,True],[True,False]])
exo_denominateur = ...  # Remplacer les points par votre réponse.
if exo_denominateur is Ellipsis or exo_denominateur is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert int(exo_denominateur) == 6
    exo_mse=(exo_batch.square()*exo_masque.unsqueeze(-1)).sum()/exo_denominateur
    torch.testing.assert_close(exo_mse,torch.tensor(5656/6))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Trois trames sont réelles, chacune avec deux erreurs carrées. Comptez les valeurs, pas seulement les trames.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_batch = torch.tensor([[[2.],[4.]],[[6.],[0.]]])
exo_masque = torch.tensor([[True,True],[True,False]])
exo_validite = exo_masque.unsqueeze(-1)
exo_moyennes = (exo_batch*exo_validite).sum(dim=1)/exo_validite.sum(dim=1)
if exo_moyennes is Ellipsis or exo_moyennes is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(exo_moyennes,torch.tensor([[3.],[6.]]))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_batch = torch.tensor([[[2.,20.],[4.,40.]],[[6.,60.],[0.,0.]]])
exo_masque = torch.tensor([[True,True],[True,False]])
exo_denominateur = exo_masque.sum()*exo_batch.shape[-1]
if exo_denominateur is Ellipsis or exo_denominateur is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert int(exo_denominateur) == 6
    exo_mse=(exo_batch.square()*exo_masque.unsqueeze(-1)).sum()/exo_denominateur
    torch.testing.assert_close(exo_mse,torch.tensor(5656/6))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [22]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()